# BERT-clean fine-tune — no manual augmentation

**Ablation control variant for `bert_augmented_weighted.ipynb`.**

Identical hyper-parameters and class-weighting, but trained **only on the Jigsaw `train.csv`**
(no `manual_200` rows mixed in). Designed so that comparing this model to
`bert_augmented_weighted` isolates the contribution of data augmentation.

Protocol (kept in lock-step with the augmented variant):
- model: `bert-base-uncased`, max_length 128
- optimizer: AdamW, lr 2e-5, weight_decay 0.01
- schedule: 2 epochs, batch size 16, seed 42
- loss: weighted cross-entropy, weights computed with
  `compute_class_weight("balanced")` on the ORIGINAL train_df
- evaluation: full `manual_annotation_200.csv` (232 rows) — *never seen during training*

## How to run

**Colab (recommended):** open this notebook, `Runtime -> Change runtime type -> T4 GPU`,
then `Run all`. The first cell auto-detects Colab, mounts Drive, and points paths at
`My Drive/6713groupwork/`. Expected wall-clock: ~40 min on T4.

**Local:** drop this file into `submission/CODE/` (where it already lives) and
`jupyter lab` from `submission/`. Data is read from `../MISC/data/` and the
trained model is saved under `../MISC/saved_model/bert_clean_model/`.

In [2]:
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # Install pinned deps if Colab image is too stale.
    os.system("pip -q install 'transformers>=4.40' 'datasets>=2.19' 'accelerate>=0.30'")
    PROJECT_ROOT = Path("/content/drive/MyDrive/6713groupwork")
    # Drive layout: transformer-friendly light-cleaned 6w split + manual annotation CSV.
    DATA_DIR = PROJECT_ROOT / "cleaned_data" / "pre-trained model" / "6w data"
    MANUAL_PATH = PROJECT_ROOT / "cleaned_data" / "new_200.csv"
    OUTPUT_DIR = PROJECT_ROOT / "outputs" / "bert_clean"
    MODEL_DIR = PROJECT_ROOT / "models"
else:
    PROJECT_ROOT = Path.cwd()
    DATA_DIR = PROJECT_ROOT.parent / "MISC" / "data"
    MANUAL_PATH = DATA_DIR / "manual_annotation_200.csv"
    OUTPUT_DIR = PROJECT_ROOT.parent / "MISC" / "outputs"
    MODEL_DIR = PROJECT_ROOT.parent / "MISC" / "saved_model"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("In Colab        :", IN_COLAB)
print("Project root    :", PROJECT_ROOT)
print("Data directory  :", DATA_DIR)
print("Manual CSV      :", MANUAL_PATH)
print("Output directory:", OUTPUT_DIR)
print("Model directory :", MODEL_DIR)
print("Device          :", DEVICE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
In Colab        : True
Project root    : /content/drive/MyDrive/6713groupwork
Data directory  : /content/drive/MyDrive/6713groupwork/cleaned_data/pre-trained model/6w data
Manual CSV      : /content/drive/MyDrive/6713groupwork/cleaned_data/new_200.csv
Output directory: /content/drive/MyDrive/6713groupwork/outputs/bert_clean
Model directory : /content/drive/MyDrive/6713groupwork/models
Device          : cuda


## 1. Load datasets

**Key difference vs. `bert_augmented_weighted.ipynb`:** we load `manual_annotation_200.csv`
**only for evaluation** — it never enters `train_dataset`.

In [3]:
def _read_csv_any_encoding(path: Path) -> pd.DataFrame:
    """Handle both UTF-8 and Latin-1 source files (manual_200 sometimes has cp1252)."""
    for enc in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"Could not decode {path} with any known encoding")

TEXT_COL = "cleaned_text"
LABEL_COL = "toxic"

train_df = _read_csv_any_encoding(DATA_DIR / "train.csv")
val_df = _read_csv_any_encoding(DATA_DIR / "val.csv")
test_df = _read_csv_any_encoding(DATA_DIR / "test.csv")
manual_df = _read_csv_any_encoding(MANUAL_PATH)

for name, df in [("train", train_df), ("val", val_df),
                 ("test", test_df), ("manual_200", manual_df)]:
    df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")
    print(f"{name:>11}: n={len(df):>6}  toxic%={df[LABEL_COL].mean():.3f}")

# manual_200 is *held out* for evaluation. Assert we are not about to leak it.
assert len(train_df) > 40_000, "train_df looks too small — did you accidentally swap datasets?"
print("\nTrain split is clean (no manual_200 rows mixed in).")

      train: n= 51128  toxic%=0.092
        val: n=  6391  toxic%=0.092
       test: n=  6391  toxic%=0.092
 manual_200: n=   232  toxic%=0.379

Train split is clean (no manual_200 rows mixed in).


## 2. Tokenize with `bert-base-uncased`

In [4]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def _to_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(
        df[[TEXT_COL, LABEL_COL]].rename(columns={LABEL_COL: "labels"}),
        preserve_index=False,
    )
    return ds

def _tokenize(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_ds = _to_hf_dataset(train_df).map(_tokenize, batched=True, remove_columns=[TEXT_COL])
val_ds = _to_hf_dataset(val_df).map(_tokenize, batched=True, remove_columns=[TEXT_COL])
test_ds = _to_hf_dataset(test_df).map(_tokenize, batched=True, remove_columns=[TEXT_COL])
manual_ds = _to_hf_dataset(manual_df).map(_tokenize, batched=True, remove_columns=[TEXT_COL])

print(train_ds)
print(val_ds)
print(test_ds)
print(manual_ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/51128 [00:00<?, ? examples/s]

Map:   0%|          | 0/6391 [00:00<?, ? examples/s]

Map:   0%|          | 0/6391 [00:00<?, ? examples/s]

Map:   0%|          | 0/232 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 51128
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 6391
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 6391
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 232
})


## 3. Class weights (balanced, from clean `train_df`)

The augmented variant computes weights on `train_df_augmented`
(which adds 174 `manual_200` rows). Here we compute on the **original**
`train_df` only, giving weights ~`[0.551, 5.407]` — same as RoBERTa / DistilBERT.

In [5]:
from sklearn.utils.class_weight import compute_class_weight

y_train = train_df[LABEL_COL].values
class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=y_train,
)
CLASS_WEIGHTS = torch.tensor(class_weights_np, dtype=torch.float32, device=DEVICE)
print("Class weights (non-toxic, toxic):", CLASS_WEIGHTS.tolist())

Class weights (non-toxic, toxic): [0.5509482622146606, 5.406937599182129]


## 4. Fine-tune with weighted cross-entropy

Same `WeightedLossTrainer` pattern as the augmented variant — only the
training dataset and the class-weight source data differ.

In [6]:
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def _compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0,
    )
    p_tox, r_tox, f1_tox, _ = precision_recall_fscore_support(
        labels, preds, average="binary", pos_label=1, zero_division=0,
    )
    try:
        probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
        auc = roc_auc_score(labels, probs)
    except Exception:
        auc = float("nan")
    return {
        "accuracy": acc,
        "f1_macro": f1_macro,
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_toxic": f1_tox,
        "precision_toxic": p_tox,
        "recall_toxic": r_tox,
        "roc_auc": auc,
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "bert_clean_results"),
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=200,
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# transformers >=4.46 renamed the Trainer kwarg `tokenizer` -> `processing_class`.
# Use inspect to stay compatible with both old and new Colab images.
import inspect as _inspect
_trainer_params = set(_inspect.signature(WeightedLossTrainer.__init__).parameters)
_tok_kwarg = "processing_class" if "processing_class" in _trainer_params else "tokenizer"

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=_compute_metrics,
    **{_tok_kwarg: tokenizer},
)

print("Starting training (2 epochs, batch=16, lr=2e-5, seed=42)...")
train_result = trainer.train()
print("Training done:", train_result.metrics)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training (2 epochs, batch=16, lr=2e-5, seed=42)...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro,F1 Toxic,Precision Toxic,Recall Toxic,Roc Auc
1,0.297148,0.335023,0.959631,0.877312,0.885158,0.869865,0.776817,0.794690,0.759729,0.975181
2,0.224280,0.388048,0.960100,0.882654,0.877869,0.887599,0.787323,0.776316,0.798646,0.974220


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Training done: {'train_runtime': 2356.7506, 'train_samples_per_second': 43.389, 'train_steps_per_second': 2.712, 'total_flos': 6670658697652320.0, 'train_loss': 0.29317026054754725, 'epoch': 2.0}


## 5. Save the best checkpoint

In [7]:
save_dir = MODEL_DIR / "bert_clean_model"
save_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))
print("Final model saved to:", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to: /content/drive/MyDrive/6713groupwork/models/bert_clean_model


## 6. Evaluate on val / test / manual_200 (full 232)

Threshold 0.5 across the board — matches the RoBERTa / DistilBERT protocol
and the "main table" entries we use in the report.

In [8]:
def _evaluate_split(name: str, dataset: Dataset) -> dict:
    out = trainer.predict(dataset)
    metrics = _compute_metrics((out.predictions, out.label_ids))
    metrics = {"split": name, "n": len(dataset), **metrics}
    return metrics

rows = [
    _evaluate_split("val", val_ds),
    _evaluate_split("test_6w", test_ds),
    _evaluate_split("manual_200", manual_ds),
]

results_df = pd.DataFrame(rows)
results_df.insert(0, "model", "BERT-clean")
print(results_df.round(4).to_string(index=False))

     model      split    n  accuracy  f1_macro  precision_macro  recall_macro  f1_toxic  precision_toxic  recall_toxic  roc_auc
BERT-clean        val 6391    0.9601    0.8827           0.8779        0.8876    0.7873           0.7763        0.7986   0.9742
BERT-clean    test_6w 6391    0.9598    0.8816           0.8773        0.8859    0.7853           0.7756        0.7953   0.9782
BERT-clean manual_200  232    0.7069    0.6400           0.7147        0.6402    0.4848           0.7273        0.3636   0.6787


## 7. Persist metrics for the cross-model comparison table

In [9]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_csv = OUTPUT_DIR / "bert_clean_summary_metrics.csv"
results_df.round(4).to_csv(out_csv, index=False)
print("Wrote:", out_csv)

# Also emit a row-per-metric view in the same shape as `roberta_base_summary_metrics.csv`
# so that downstream comparison scripts can concat all four models uniformly.
pivot = (
    results_df.set_index("split")[
        ["accuracy", "f1_macro", "precision_macro", "recall_macro",
         "f1_toxic", "precision_toxic", "recall_toxic", "roc_auc"]
    ]
    .T
)
pivot_csv = OUTPUT_DIR / "bert_clean_summary_metrics_wide.csv"
pivot.round(4).to_csv(pivot_csv)
print("Wrote:", pivot_csv)

Wrote: /content/drive/MyDrive/6713groupwork/outputs/bert_clean/bert_clean_summary_metrics.csv
Wrote: /content/drive/MyDrive/6713groupwork/outputs/bert_clean/bert_clean_summary_metrics_wide.csv


## Sanity-check vs. the augmented variant

After training finishes, the expected pattern for a **correct** clean run is:

| split       | BERT-clean    | BERT-aug-w (reported) | note                                      |
|-------------|---------------|-----------------------|-------------------------------------------|
| val         | ~0.78 F1_tox  | 0.7901 F1_tox         | small drop, both trained on 6w Jigsaw     |
| test_6w     | ~0.78 F1_tox  | 0.7824 F1_tox         | nearly identical — Jigsaw is the same set |
| manual_200  | ~0.50 F1_tox  | 0.7368 F1_tox (n=58)  | **big gap — that gap IS the ablation**    |

If `manual_200` F1_toxic comes out close to BERT-aug-w's 0.7368, something leaked
(e.g. `train_df` contains rows from `manual_annotation_200.csv`). In that case
double-check the data-loading cell above.